In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configurare la mitigazione degli errori

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** La versione beta di un nuovo modello di esecuzione è ora disponibile. Il modello di esecuzione diretto offre maggiore flessibilità nella personalizzazione del tuo workflow di mitigazione degli errori. Consulta la guida al [Modello di esecuzione diretto](/guides/directed-execution-model) per ulteriori informazioni.


<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit-ibm-runtime~=0.43.1
```
</details>

Le tecniche di mitigazione degli errori permettono agli utenti di ridurre gli errori dei circuiti modellando il rumore del dispositivo al momento dell'esecuzione. Questo comporta tipicamente un overhead di pre-elaborazione quantistica legato all'addestramento del modello e un overhead di post-elaborazione classica per mitigare gli errori nei risultati grezzi tramite il modello generato.

La primitiva Estimator supporta diverse tecniche di mitigazione degli errori, tra cui [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec) e [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Consulta [Tecniche di mitigazione e soppressione degli errori](error-mitigation-and-suppression-techniques) per una spiegazione di ciascuna. Quando usi le primitive, puoi attivare o disattivare i singoli metodi. Consulta la sezione [Impostazioni personalizzate degli errori](#advanced-error) per i dettagli.

> **Note:** Sampler non supporta la mitigazione degli errori, ma puoi usare il pacchetto [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (matrix-free measurement mitigation) per eseguire la mitigazione degli errori localmente.

Estimator supporta anche `resilience_level`. Il livello di resilienza specifica quanto rafforzare la resistenza agli errori. Livelli più alti producono risultati più accurati, a scapito di tempi di elaborazione più lunghi. I livelli di resilienza possono essere usati per configurare il compromesso costo/accuratezza quando si applica la mitigazione degli errori alle query delle primitive. La mitigazione degli errori riduce gli errori (bias) nei risultati elaborando gli output di una collezione, o ensemble, di circuiti correlati. Il grado di riduzione degli errori dipende dal metodo applicato. Il livello di resilienza astrae la scelta dettagliata del metodo di mitigazione degli errori per consentire agli utenti di ragionare sul compromesso costo/accuratezza più adatto alla loro applicazione.

Detto questo, ogni livello corrisponde a uno o più metodi con un livello crescente di overhead di campionamento quantistico, per permetterti di sperimentare diversi compromessi tra tempo e accuratezza. La tabella seguente mostra quali livelli e metodi corrispondenti sono disponibili per ciascuna delle primitive.

> **Info:** La mitigazione degli errori è specifica per il task, quindi le tecniche che puoi applicare variano a seconda che tu stia campionando una distribuzione o calcolando valori di aspettazione.

<span id="resilience-table"></span>

Estimator supporta i seguenti livelli di resilienza. Sampler non supporta i livelli di resilienza.

| Livello di resilienza | Definizione                                                                                                         | Tecnica                                                                             |
|-----------------------|---------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------|
| 0                     | Nessuna mitigazione                                                                                                 | Nessuna                                                                             |
| 1 [Predefinito]       | Costi di mitigazione minimi: mitiga gli errori associati agli errori di readout                                     | Twirled Readout Error eXtinction (TREX) con measurement twirling                   |
| 2                     | Costi di mitigazione medi. Riduce tipicamente il bias negli stimatori, ma non garantisce risultati privi di bias.   | Livello 1 + Zero Noise Extrapolation (ZNE) e gate twirling

> **Info:** I livelli di resilienza sono attualmente in beta, quindi l'overhead di campionamento e la qualità della soluzione varieranno da circuito a circuito. Nuove funzionalità, opzioni avanzate e strumenti di gestione saranno rilasciati progressivamente. Non è garantito che specifici metodi di mitigazione degli errori vengano applicati a ogni livello di resilienza.

## Configurare Estimator con i livelli di resilienza
Puoi usare i livelli di resilienza per specificare le tecniche di mitigazione degli errori, oppure impostare tecniche personalizzate individualmente come descritto in [Impostazioni personalizzate degli errori.](#advanced-error)

<details>
<summary>Livello di resilienza 0</summary>

Nessuna mitigazione degli errori viene applicata al programma dell'utente.

</details>

<details>
<summary>Livello di resilienza 1</summary>

Il livello 1 applica la **mitigazione degli errori di readout** e il **measurement twirling** tramite una tecnica model-free nota come Twirled Readout Error eXtinction (TREX). Riduce l'errore di misura diagonalizzando il canale di rumore associato alla misura, ribaltando casualmente i qubit tramite gate X immediatamente prima della misura. Un termine di riscalatura dal canale di rumore diagonale viene appreso eseguendo il benchmarking di circuiti casuali inizializzati nello stato zero. Ciò consente al servizio di rimuovere il bias dai valori di aspettazione causato dal rumore di readout. Questo approccio è descritto ulteriormente in [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Livello di resilienza 2</summary>

Il livello 2 applica le **tecniche di mitigazione degli errori incluse nel livello 1** e applica inoltre il **gate twirling** e usa il **metodo Zero Noise Extrapolation (ZNE)**. ZNE calcola il valore di aspettazione dell'osservabile per diversi fattori di rumore (fase di amplificazione) e poi usa i valori di aspettazione misurati per inferire il valore di aspettazione ideale al limite di rumore zero (fase di estrapolazione). Questo approccio tende a ridurre gli errori nei valori di aspettazione, ma non garantisce un risultato privo di bias.

![Questa immagine mostra un grafico. L'asse x è etichettato come Fattore di amplificazione del rumore. L'asse y è etichettato come Valore di aspettazione. Una linea con pendenza crescente è etichettata come Valore mitigato. I punti vicini alla linea sono valori amplificati dal rumore. C'è una linea orizzontale appena sopra l'asse X etichettata come Valore esatto.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Illustrazione del metodo ZNE")

L'overhead di questo metodo scala con il numero di fattori di rumore. Le impostazioni predefinite campionano il valore di aspettazione a tre fattori di rumore, con un overhead di circa 3x quando si utilizza questo livello di resilienza.

Nel livello 2, il metodo TREX ribalta casualmente i qubit tramite gate X immediatamente prima della misura e ribalta il bit misurato corrispondente se è stato applicato un gate X. Questo approccio è descritto ulteriormente in [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Esempio
L'interfaccia `EstimatorV2` permette agli utenti di lavorare agevolmente con la varietà di metodi di mitigazione degli errori per ridurre gli errori nei valori di aspettazione degli osservabili. Il codice seguente usa la Zero Noise Extrapolation e la mitigazione degli errori di readout semplicemente impostando `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Impostazioni personalizzate degli errori
Puoi attivare e disattivare i singoli metodi di mitigazione e soppressione degli errori, tra cui il dynamical decoupling, il gate e measurement twirling, la mitigazione degli errori di misura, PEC e ZNE. Consulta [Tecniche di mitigazione e soppressione degli errori](error-mitigation-and-suppression-techniques) per una spiegazione di ciascuno.

> **Note:** - Non tutte le opzioni sono disponibili per entrambe le primitive. Consulta la [tabella delle opzioni disponibili](runtime-options-overview#options-table) per l'elenco delle opzioni disponibili.
> - Non tutti i metodi funzionano insieme su tutti i tipi di circuiti. Consulta la [tabella di compatibilità delle funzionalità](runtime-options-overview#options-compatibility-table) per i dettagli.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">